In [1]:
# روباستنس



import pandas as pd
from linearmodels.panel import PanelOLS

# 1. Load Data
file_path = 'panel_data_149_EPI_TII_filled_logGDPpc.xlsx'
df = pd.read_excel(file_path)

# 2. Set Panel Data Structure (MultiIndex: Country and Year)
df = df.set_index(['Country', 'Year'])

# 3. Create 2-Year Lagged Variables for Robustness Check (t-2)
lag_vars = ['rl', 'rq', 'pv', 'cc', 'ge', 'va', 'TII', 'log_GDPpc']
for var in lag_vars:
    df[f'{var}_lag2'] = df.groupby(level='Country')[var].shift(2)

# Drop rows with NaN values created by lagging (years 2015 and 2016 will be dropped)
df_clean = df.dropna(subset=[f'{var}_lag2' for var in lag_vars] + ['cc', 'ge', 'va', 'EPI'])

# 4. Define and Run the TWFE Models with 2-Year Lags

# --- Model 1: Control of Corruption (CC_t) ---
exog_m1 = df_clean[['rl_lag2', 'log_GDPpc_lag2', 'TII_lag2']]
model1 = PanelOLS(df_clean['cc'], exog_m1, entity_effects=True, time_effects=True)
results1 = model1.fit(cov_type='robust')
print("=== Model 1 (t-2): Predictors of Control of Corruption (CC) ===")
print(results1.summary)
print("\n" + "="*80 + "\n")

# --- Model 2: Government Effectiveness (GE_t) ---
exog_m2 = df_clean[['rq_lag2', 'cc_lag2', 'pv_lag2', 'log_GDPpc_lag2', 'TII_lag2']]
model2 = PanelOLS(df_clean['ge'], exog_m2, entity_effects=True, time_effects=True)
results2 = model2.fit(cov_type='robust')
print("=== Model 2 (t-2): Predictors of Government Effectiveness (GE) ===")
print(results2.summary)
print("\n" + "="*80 + "\n")

# --- Model 3: Voice and Accountability (VA_t) ---
exog_m3 = df_clean[['ge_lag2', 'cc_lag2', 'log_GDPpc_lag2', 'TII_lag2']]
model3 = PanelOLS(df_clean['va'], exog_m3, entity_effects=True, time_effects=True)
results3 = model3.fit(cov_type='robust')
print("=== Model 3 (t-2): Predictors of Voice and Accountability (VA) ===")
print(results3.summary)
print("\n" + "="*80 + "\n")

# --- Model 4: E-Participation Index (EPI_t) - THE BOTTLENECK ---
exog_m4 = df_clean[['va_lag2', 'ge_lag2', 'cc_lag2', 'log_GDPpc_lag2', 'TII_lag2']]
model4 = PanelOLS(df_clean['EPI'], exog_m4, entity_effects=True, time_effects=True)
results4 = model4.fit(cov_type='robust')
print("=== Model 4 (t-2): Predictors of E-Participation (EPI) ===")
print(results4.summary)


=== Model 1 (t-2): Predictors of Control of Corruption (CC) ===
                          PanelOLS Estimation Summary                           
Dep. Variable:                     cc   R-squared:                        0.1580
Estimator:                   PanelOLS   R-squared (Between):              0.9345
No. Observations:                1192   R-squared (Within):               0.0932
Date:                Tue, Mar 31 2026   R-squared (Overall):              0.9331
Time:                        14:49:40   Log-likelihood                   -2425.1
Cov. Estimator:                Robust                                           
                                        F-statistic:                      64.590
Entities:                         149   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                  F(3,1033)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-sta